In [1]:
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import os
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

import pvlib
from pvlib import location, pvsystem, modelchain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS
import re


In [2]:
# -----------------------------------------------------------------------------
# 1. Data Loading & Generation (Same as Mamba)
# -----------------------------------------------------------------------------


import pandas as pd
import re
import numpy as np
from pvlib import location, pvsystem, modelchain
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS

def get_station_metadata(station_num, df):
    """
    Retrieves and parses metadata for a specific station from the DataFrame.
    
    Args:
        station_num (int or str): The station number (e.g., 5 or '05').
        df (pd.DataFrame): The source dataframe containing station metadata.
        
    Returns:
        dict: A clean dictionary with numerical values ready for simulation.
    """
    
    # 1. Format the station ID
    station_id_str = f"station{str(station_num).zfill(2)}"
    
    # 2. Filter the dataframe
    station_row = df[df['Station_ID'] == station_id_str]
    
    if station_row.empty:
        raise ValueError(f"Station ID {station_id_str} not found in DataFrame.")
    
    # Get the single row as a Series
    row = station_row.iloc[0]

    # --- Helper 1: Parse "Key:Value" blocks (lines separated by \n) ---
    def parse_kv_string(text_block):
        data = {}
        if isinstance(text_block, str):
            for item in text_block.split('\n'):
                if ':' in item:
                    k, v = item.split(':', 1)
                    data[k.strip()] = v.strip()
        return data

    # --- Helper 2: Extract numeric values safely (e.g., "250 Wp" -> 250.0) ---
    def extract_num(val, default=0.0):
        if pd.isna(val): return default
        match = re.search(r"[-+]?\d*\.\d+|\d+", str(val))
        return float(match.group()) if match else default

    # 3. Parse complex text columns into dictionaries
    module_items = parse_kv_string(row.get('Module', ''))
    inverter_items = parse_kv_string(row.get('Inverters', ''))
    layout_items = parse_kv_string(row.get('Layout', ''))

    # 4. Build the final clean dictionary
    # We use extract_num to ensure we pass numbers (floats/ints) to the model, not strings.
    metadata = {
        # --- Identity & Location ---
        'Station_ID': station_id_str,
        'Longitude': float(row['Longitude']),
        'Latitude': float(row['Latitude']),
        'Array_Tilt': row['Array_Tilt'], # Kept as string (e.g., "South 30") for parsing in main model
        
        # --- System Capacity & Dimensions ---
        'Capacity': float(row['Capacity']),          # Total Station Capacity (kW)
        'Panel_Size': float(row.get('Panel_Size', 1.62)), # Panel Area in m^2
        'Total_Panel_Number': int(row.get('Panel_Number', 0)),
        'PV_Technology': row['PV_Technology'],

        # --- Extracted Module Specs (Cleaned to Float) ---
        'Module_Pmax': extract_num(module_items.get('Pmax'), default=250),
        'Module_Vmpp': extract_num(module_items.get('Vmpp'), default=30),
        'Module_Impp': extract_num(module_items.get('Impp'), default=8),
        
        # --- Extracted Inverter Specs (Cleaned to Float) ---
        'Inverter_Rated_Power': extract_num(inverter_items.get('Rated power'), default=500),
        'Inverter_Max_DC_Voltage': extract_num(inverter_items.get('Max. DC voltage'), default=1000),
        
        # --- Extracted Layout Specs (Cleaned to Int) ---
        # Note: Layout strings can look like "modules per string:20"
        'Modules_per_String': int(extract_num(layout_items.get('modules per string'), default=20)),
        'Strings_per_Inverter': int(extract_num(layout_items.get('strings per inverter'), default=100)),
    }
    
    return metadata


def calculate_clearsky_indices(metadata, df):
    """
    Calculates both Irradiance (K_CS) and Power (K_PV) Clear Sky Indices.
    
    Args:
        metadata (dict): Parsed station metadata.
        df (pd.DataFrame): Time-series data with 'lmd_totalirrad' and 'power' columns.
                           'power' should be in MW (or kW, logic adapts).
                           'lmd_totalirrad' should be in W/m^2.
    
    Returns:
        pd.DataFrame: Original df with added columns 'GHI_clr', 'K_CS', 'P_CLR', 'K_PV'.
    """
    
    print(f"--- Processing {metadata.get('Station_ID', 'Unknown')} ---")
    
    # ---------------------------------------------------------
    # 1. SETUP LOCATION & TIME
    # ---------------------------------------------------------
    # The paper explicitly states timestamps are in Coordinated Universal Time (UTC).
    # Setting tz='UTC' ensures pvlib aligns Solar Noon correctly (approx 04:00 UTC for China).
    lat = metadata['Latitude']
    lon = metadata['Longitude']
    
    site = location.Location(lat, lon, tz='UTC')
    
    # Ensure index is Datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df['date_time'] = pd.to_datetime(df['date_time'])
        df.set_index('date_time', inplace=True)

    # ---------------------------------------------------------
    # 2. CALCULATE CLEAR SKY IRRADIANCE (GHI_clr)
    # ---------------------------------------------------------
    print("Calculating Clear Sky Irradiance (Ineichen model)...")
    
    # Calculate theoretical GHI (returns W/m^2)
    cs = site.get_clearsky(df.index, model='ineichen')
    df['GHI_clr'] = cs['ghi']

    # --- Index 1: Irradiance Clear Sky Index (K_CS) ---
    # Formula: K_CS = Measured GHI / Clear Sky GHI
    
    if 'lmd_totalirrad' in df.columns:
        # Standardize Measured Units: 
        # lmd_totalirrad is W/m^2.
        meas_ghi = df['lmd_totalirrad']
        

        # Calculate Index (Filter: Only when model expects sun > 10 W/m^2)
        # Naive filter to avoid night time noise
        df['K_CS_day'] = np.where(
            df['GHI_clr'] > 10, 
            meas_ghi / df['GHI_clr'], 
            0.0
        )
        df['K_CS_dayNight'] = meas_ghi / df['GHI_clr']

        df['K_CS'] = df['K_CS_day']
        
        # Clip outliers (cloud edge effects can cause > 1.0)
        df['K_CS'] = df['K_CS'].clip(lower=0.0, upper=1.25)
    else:
        print("Warning: 'lmd_totalirrad' column missing. K_CS not calculated.")

    # ---------------------------------------------------------
    # 3. SETUP PV SYSTEM MODEL (For P_CLR)
    # ---------------------------------------------------------
    # Extract Tilt safely from string 
    try:
        tilt_str = str(metadata.get('Array_Tilt'))
        tilt_match = re.search(r"[\d.]+", tilt_str)
        tilt = float(tilt_match.group()) if tilt_match else 33.0
    except Exception as e:
        print(f"Error extracting tilt: {e} .... Assuming 33 degrees")
        tilt = 33.0
    
    # Default Azimuth: 180 (South) for Northern Hemisphere
    azimuth = 180 

    # Define Module Parameters (Generic Poly-Si based on metadata)
    # Hard coded Module for specific station 07
    module_params = pvsystem.retrieve_sam('CECMod')['Yingli_Energy__China__YL250P_29b']
    # Module Parameters can be retreived from pvsystem.retrieve_sam by defining the module name and for now for simplicity we use hardcoded one.
    # module_params = {
    #     'pdc0': metadata['Module_Pmax'],   # STC Power (Watts)
    #     'gamma_pdc': -0.004,               # Temp coeff
    #     'b': 0.05, 'a': -3.56              # Sandia model defaults
    # }
    
    # Define Inverter Parameters (One Block)
    # Block Size = Modules per string * Strings per inverter * Module Power
    block_dc_watts = (metadata['Modules_per_String'] * metadata['Strings_per_Inverter'] * metadata['Module_Pmax'])
    

    # The same with Inverter infoirmation i used hardcoded info for station 07    
    # inv_params = pvsystem.retrieve_sam("cecinverter")

    ivt_para = pvsystem.retrieve_sam('cecinverter')['Advanced_Energy_Industries__Solaron_500kW__3159500_XXXX___480V_']
    ivt_para["Pdco"], ivt_para['Vdco'], ivt_para["Vdcmax"], ivt_para['Idcmax'], = 567000, 315, 1000, 1134
    ivt_para["Mppt_low"], ivt_para['Mppt_high'] = 460, 950
    inverter_parameters = ivt_para
    
    # Temperature Model (Open Rack)
    temp_model = TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
    
    system = pvsystem.PVSystem(
        surface_tilt=tilt,
        surface_azimuth=azimuth,
        module_parameters=module_params,
        inverter_parameters=inverter_parameters,
        temperature_model_parameters=temp_model,
        modules_per_string=metadata['Modules_per_String'],
        strings_per_inverter=metadata['Strings_per_Inverter']
        # modules_per_string=20, strings_per_inverter=100 # Must be retreived from metadata, but for now hardcoded
    )

    # ---------------------------------------------------------
    # 4. RUN MODEL CHAIN FOR POWER
    # ---------------------------------------------------------
    print("Running PV Power Simulation...")

    mc = ModelChain(system, site, transposition_model='perez',
                        solar_position_method='nrel_numpy',
                        aoi_model='physical', spectral_model='no_loss')

    # mc = modelchain.ModelChain(
    #     system, 
    #     site, 
    #     aoi_model='physical', 
    #     spectral_model='no_loss', 
    #     ac_model='pvwatts'
    # )

    # Run using the Clear Sky weather data
    mc.run_model(cs)

    # ---------------------------------------------------------
    # 5. SCALE & CALCULATE POWER INDEX (K_PV)
    # ---------------------------------------------------------
    # The simulation (mc.results.ac) is for ONE Block. Scale to full station.
    
    station_total_capacity_watts = metadata['Capacity'] * 1000  # kW -> Watts [cite: 109]
    scaling_factor = station_total_capacity_watts / block_dc_watts
    
    # Calculate Station Clear Sky Power (MW)
    # FillNa(0) handles night time; Divide by 1M to get MW
    p_clr_watts = mc.results.ac.fillna(0) * scaling_factor
    df['P_CLR'] = p_clr_watts / 1_000_000 

    # --- Index 2: Power Clear Sky Index (K_PV) ---
    # Formula: K_PV = Measured Power / Clear Sky Power
    
    if 'power' in df.columns:
        meas_power = df['power']
        
        # Check units: Paper says Power is MW[cite: 109].
        # Heuristic: If max > 500, it is likely kW, so divide by 1000.
        if meas_power.max() > 500:
            meas_power = meas_power / 1000.0
            
        # Calculate Index
        # Filter: Only calculate when model expects > 0.05 MW to avoid night noise
        df['K_PV'] = np.where(
            df['P_CLR'] > 1.0, 
            meas_power / df['P_CLR'], 
            0.0
        )
        
        # Clip for cleanliness (Values > 1.0 mean better than clear sky or model error)
        df['K_PV'] = df['K_PV'].clip(lower=0.0, upper=1.5)
    else:
        print("Warning: 'power' column missing. K_PV not calculated.")

    return df


def calculate_nwp_power(metadata, df):
    """
    Calculates the expected PV Power output (MW) based purely on NWP Weather Forecasts.
    This acts as a "Physics-Based Baseline" to compare against the AI model.
    
    Args:
        metadata (dict): Station metadata from get_station_metadata()
        df (pd.DataFrame): DataFrame containing 'nwp_' columns.
        
    Returns:
        pd.Series: The calculated power in MW, aligned with df.index.
    """
    station_id = metadata.get('Station_ID', 'Unknown')
    print(f"--- Calculating NWP Physical Power for {station_id} ---")
    
    # ---------------------------------------------------------
    # 1. SETUP LOCATION & TIME
    # ---------------------------------------------------------
    lat = metadata['Latitude']
    lon = metadata['Longitude']
    
    # Define Location (UTC to align with solar position)
    site = location.Location(lat, lon, tz='UTC')
    
    # Ensure DataFrame index is Datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # ---------------------------------------------------------
    # 2. PREPARE WEATHER DATA (Map NWP -> PVLib Standard)
    # ---------------------------------------------------------
    # PVLib ModelChain expects columns: 'ghi', 'dni', 'dhi', 'temp_air', 'wind_speed', 'pressure'
    weather_nwp = pd.DataFrame(index=df.index)
    
    # Map available columns
    weather_nwp['ghi'] = df['nwp_globalirrad']
    weather_nwp['dni'] = df['nwp_directirrad']
    weather_nwp['temp_air'] = df['nwp_temperature']
    weather_nwp['wind_speed'] = df['nwp_windspeed']
    
    # Handle Pressure (Default to 101325 Pa if missing)
    if 'nwp_pressure' in df.columns:
        # Assumption: NWP pressure is often hPa, PVLib expects Pa. 
        # If your data is already Pa, remove the * 100.
        weather_nwp['pressure'] = df['nwp_pressure'] * 100 
    else:
        weather_nwp['pressure'] = 101325.0

    # CALCULATE MISSING DHI (Diffuse Horizontal Irradiance)
    # Physics Relation: GHI = DNI * cos(Zenith) + DHI
    # Therefore: DHI = GHI - (DNI * cos(Zenith))
    
    # Get Solar Position (Zenith) for every timestamp
    solpos = site.get_solarposition(weather_nwp.index)
    zenith_rad = np.radians(solpos['zenith'])
    
    # Calculate DHI
    weather_nwp['dhi'] = weather_nwp['ghi'] - (weather_nwp['dni'] * np.cos(zenith_rad))
    
    # Physics Check: Irradiance cannot be negative.
    # (Negative values happen when Sun is low and NWP GHI/DNI are slightly mismatched)
    weather_nwp['dhi'] = weather_nwp['dhi'].clip(lower=0.0)
    weather_nwp['dni'] = weather_nwp['dni'].clip(lower=0.0)
    weather_nwp['ghi'] = weather_nwp['ghi'].clip(lower=0.0)

    # ---------------------------------------------------------
    # 3. DEFINE PV SYSTEM (Hardware Specs)
    # ---------------------------------------------------------
    # Extract Tilt
    try:
        tilt_str = str(metadata.get('Array_Tilt'))
        tilt_match = re.search(r"[\d.]+", tilt_str)
        tilt = float(tilt_match.group()) if tilt_match else 33.0
    except:
        tilt = 33.0
        
    # Hardcoded/Standardized Equipment (Matching your previous Setup)
    # Module: Yingli YL250P-29b
    module_params = pvsystem.retrieve_sam('CECMod')['Yingli_Energy__China__YL250P_29b']
    
    # Inverter: Advanced Energy 500kW
    ivt_para = pvsystem.retrieve_sam('cecinverter')['Advanced_Energy_Industries__Solaron_500kW__3159500_XXXX___480V_']
    # Override generic params with specific constraints if needed
    ivt_para["Pdco"], ivt_para['Vdco'], ivt_para["Vdcmax"], ivt_para['Idcmax'] = 567000, 315, 1000, 1134
    ivt_para["Mppt_low"], ivt_para['Mppt_high'] = 460, 950
    
    # Temp Model
    temp_model = TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
    
    # Create the System Object
    system = pvsystem.PVSystem(
        surface_tilt=tilt,
        surface_azimuth=180, # Facing South
        module_parameters=module_params,
        inverter_parameters=ivt_para,
        temperature_model_parameters=temp_model,
        modules_per_string=metadata['Modules_per_String'],
        strings_per_inverter=metadata['Strings_per_Inverter']
    )

    # ---------------------------------------------------------
    # 4. RUN PHYSICAL SIMULATION (ModelChain)
    # ---------------------------------------------------------
    print("Running ModelChain with NWP Weather...")
    
    mc = ModelChain(system, site, 
                    transposition_model='perez',
                    solar_position_method='nrel_numpy',
                    aoi_model='physical', 
                    spectral_model='no_loss')

    # Run the simulation using the PREPARED NWP weather
    mc.run_model(weather_nwp)
    
    # ---------------------------------------------------------
    # 5. SCALE TO FULL STATION CAPACITY
    # ---------------------------------------------------------
    # The simulation result (mc.results.ac) is for ONE "Inverter Block" defined above.
    # We must scale this up to match the total Station Capacity.
    
    # Calculate DC Watts of one defined block
    block_dc_watts = (metadata['Modules_per_String'] * metadata['Strings_per_Inverter'] * metadata['Module_Pmax'])
    
    # Get Total Station Capacity in Watts (Capacity is usually in kW in metadata)
    station_total_capacity_watts = metadata['Capacity'] * 1000
    
    # Scaling Factor
    if block_dc_watts > 0:
        scaling_factor = station_total_capacity_watts / block_dc_watts
    else:
        scaling_factor = 1.0 # Fallback to avoid div/0
        
    # Get AC Power, fill NaNs (night), scale, and convert to MW
    # result is in Watts -> divide by 1,000,000 for MW
    nwp_power_mw = (mc.results.ac.fillna(0) * scaling_factor) / 1_000_000
    
    # Final Clip (Power cannot be negative)
    nwp_power_mw = nwp_power_mw.clip(lower=0.0)
    
    print(f"Done. Mean Predicted Power: {nwp_power_mw.mean():.4f} MW")
    
    return nwp_power_mw


In [3]:
# ==========================================================
# Real Power PV generated power data.

# read the data for PVOD
abs_path = os.path.abspath("/home/muhammadhassan/App_v02/physics_informed_inves/PVODdatasets_v1")

station07_path = os.path.join(abs_path, "station07.csv")
station07_data = pd.read_csv(station07_path)

metadata_path = os.path.join(abs_path, "metadata.csv")
metaData = pd.read_csv(metadata_path)


station07_metaData = get_station_metadata(7, metaData)
station07_data = pd.read_csv(station07_path)

In [4]:



# 3. Run Calculation
print(station07_metaData)
station07_df = calculate_clearsky_indices(station07_metaData, station07_data)

{'Station_ID': 'station07', 'Longitude': 113.64187, 'Latitude': 36.64403, 'Array_Tilt': 'South 31°', 'Capacity': 20000.0, 'Panel_Size': 1.6335, 'Total_Panel_Number': 80000, 'PV_Technology': 'Poly-Si', 'Module_Pmax': 250.0, 'Module_Vmpp': 29.8, 'Module_Impp': 8.39, 'Inverter_Rated_Power': 500.0, 'Inverter_Max_DC_Voltage': 1000.0, 'Modules_per_String': 20, 'Strings_per_Inverter': 100}
--- Processing station07 ---
Calculating Clear Sky Irradiance (Ineichen model)...
Running PV Power Simulation...


In [5]:
station07_df['NWP_Power_MW'] = calculate_nwp_power(station07_metaData, station07_df)

--- Calculating NWP Physical Power for station07 ---
Running ModelChain with NWP Weather...
Done. Mean Predicted Power: 3.2083 MW


In [6]:
def prepare_features(df):
    data = df.copy()
    
    # 1. Cyclic Encoding for Hour (0-23)
    # This helps the model know that 23:00 is close to 00:00
    data['hour_sin'] = np.sin(2 * np.pi * data.index.hour / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data.index.hour / 24)
    
    # 2. Cyclic Encoding for Day of Year (1-365)
    # This captures seasonality (Summer vs Winter)
    data['day_sin'] = np.sin(2 * np.pi * data.index.dayofyear / 365)
    data['day_cos'] = np.cos(2 * np.pi * data.index.dayofyear / 365)

    # 3. Months
    data['month_sin'] = np.sin(2 * np.pi * data.index.month / 12)
    data['month_cos'] = np.cos(2 * np.pi * data.index.month / 12)

    # seasons 
    data['season_sin'] = np.sin(2 * np.pi * data.index.month / 4)
    data['season_cos'] = np.cos(2 * np.pi * data.index.month / 4)
    
    
    
    return data


station07_df = prepare_features(station07_df)




In [7]:
import os
import numpy as np
import pandas as pd
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
from typing import List


warnings.filterwarnings('ignore')



class MultiStepDataset(Dataset):
    """
    prepare data set for the model. As X is the past F features, Y is the target and P is the prediction features represents the predictions of the corresponding time step after seq_len, 
    seq_len
    Returns:
        x_past   : [seq_len, F]
        y_future : [pred_len, 1]
        x_future : [pred_len, F]
    """
    def __init__(self, X, Y, P, seq_len=96, pred_len=96): # One day back and Forward.
        self.X = X  # numpy array [T, F]
        self.y = Y  # numpy array [T, 1] (K_PV)
        self.P = P  # numpy array [T, F]
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.X) - self.seq_len - self.pred_len + 1

    def __getitem__(self, i):
        # past window
        x_past = self.X[i : i + self.seq_len]  # [seq_len, F]

        # future horizon
        start_fut = i + self.seq_len
        end_fut   = start_fut + self.pred_len

        y_future = self.y[start_fut : end_fut]         # [pred_len, 1]
        x_future = self.P[start_fut : end_fut]         # [pred_len, F]

        return (
            torch.tensor(x_past,   dtype=torch.float32),
            torch.tensor(y_future, dtype=torch.float32),
            torch.tensor(x_future, dtype=torch.float32),
        )




def prepare_rolling_folds(
    df, input_cols, target_col, prediction_cols,
    n_splits=2, seq_len=96, pred_len=96, batch_size=32
):


    # types Validation 
    X_raw = df[input_cols].values.astype(np.float32)
    P_raw = df[prediction_cols].values.astype(np.float32)
    
    y_raw = df[target_col].values.astype(np.float32)

    # Expanding rolling window, usign retraining.
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    fold = 0
    for train_index, test_index in tscv.split(X_raw):
        fold += 1
        
        # Split Data
        X_train_fold = X_raw[train_index]
        P_train_fold = P_raw[train_index]
        y_train_fold = y_raw[train_index]

        X_test_fold = X_raw[test_index]
        P_test_fold = P_raw[test_index]
        y_test_fold = y_raw[test_index]

        # Prepend Lookback for Test Set
        # We need the last 'seq_len' points from train to predict the first point of test
        X_lookback = X_train_fold[-seq_len:]
        P_lookback = P_train_fold[-seq_len:]
        y_lookback = y_train_fold[-seq_len:]

        X_test_final = np.concatenate([X_lookback, X_test_fold], axis=0)
        P_test_final = np.concatenate([P_lookback, P_test_fold], axis=0)
        y_test_final = np.concatenate([y_lookback, y_test_fold], axis=0)

        # Create Datasets
        train_dataset = MultiStepDataset(
            X_train_fold, y_train_fold, P_train_fold, seq_len=seq_len, pred_len=pred_len
        )
        test_dataset = MultiStepDataset(
            X_test_final, y_test_final, P_test_final, seq_len=seq_len, pred_len=pred_len
        )

        # Create Loaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

        yield train_loader, test_loader




In [8]:
df = station07_df.copy(deep=True)

In [9]:
class MambaModelConfigs:
    def __init__(self):
        # --- 1. Data Dimensions ---
        self.seq_len = 5        # L: Look-back window
        self.pred_len = 3     # H: Prediction horizon

        # Define  features by name

        self.PAST_INPUT_COLS = [ 'K_PV','nwp_globalirrad']
        self.FUTURE_INPUT_COLS = ['nwp_globalirrad']
        self.TARGET_COL = ['K_PV']
        
        # --- 2. Dynamic Shadow Calculation ---
        # Identify which features exist in both Past and Future (Intersection)
        self.common_features = [f for f in self.FUTURE_INPUT_COLS if f in self.PAST_INPUT_COLS]
        self.num_shadows = len(self.common_features)
        
        # Flag to enable external prediction logic (1 = True, 0 = False)
        self.include_pred = 1 if len(self.FUTURE_INPUT_COLS) > 0 else 0

        # --- 3. Model Input Width ---
        # enc_in = Original Past Features + Shadow Features
        # The model needs this total width to initialize RevIN and Mamba layers
        self.enc_in = len(self.PAST_INPUT_COLS) + self.num_shadows
        
        self.c_out = len(self.TARGET_COL)
        
        self.kernel_size = 25  
        self.n_embed = 32     
        self.d_state = 32      # The size of the latent "hidden state". A larger d_state allows the model to remember more complex dynamics from the past
        self.dconv = 2         # Mamba uses a small local convolution to smooth the input before the state space processing, which helps capture local temporal patterns.
        self.e_fact = 2        # (Expansion Factor): This determines the internal "width" of the block. It projects the input to a higher dimension to find non-linear patterns before projecting it back
        self.dropout = 0.2

        self.n_splits = 2 
        self.batch_size = 2
        self.epochs = 100
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

configs = MambaModelConfigs()

In [10]:
fold_gen = prepare_rolling_folds(

        df, configs.PAST_INPUT_COLS, configs.TARGET_COL, configs.FUTURE_INPUT_COLS,
        n_splits=configs.n_splits,
        seq_len=configs.seq_len,
        pred_len=configs.pred_len,
        batch_size=configs.batch_size
    )

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, test_loader = next(fold_gen)
        
x_past, y_future, x_future = next(iter(train_loader))

In [12]:
x_past.shape, y_future.shape, x_future.shape

(torch.Size([2, 5, 2]), torch.Size([2, 3, 1]), torch.Size([2, 3, 1]))

In [13]:
x_past[0, : , :] # First batch of past data

tensor([[2.4547e-01, 3.3448e+02],
        [2.1110e-01, 3.2526e+02],
        [1.9176e-01, 3.1440e+02],
        [2.1862e-01, 2.9610e+02],
        [2.6863e-01, 2.6147e+02]])

In [14]:
x_future[0, : , :]

tensor([[234.7100],
        [187.6300],
        [155.6300]])

In [15]:
y_future[0, : , :] 

tensor([[0.2920],
        [0.2789],
        [0.3740]])

In [43]:
import torch
import torch.nn as nn

# Official RevIN implementation https://github.com/ts-kim/RevIN 
"""
Reversible Instance Normalization
it removes the need for the model to learn different "rules" for Summer vs. Winter.

- In solar forecasting, a clear sunny day in winter has a much lower peak (amplitude) than a clear sunny day in summer due to the sun's angle. 
Standard models struggle because they see these as two different patterns
- RevIN normalizes each input window individually: 
    - It calculates the mean (μ) and standard deviation (σ) for just that specific window of time
    - It subtracts the mean and divides by the standard deviation.
To the model, a "Winter Sunny Day" and a "Summer Sunny Day" look almost identical (they both become a standardized curve). 
The model focuses on learning the shape (the curve of the sun, cloud interruptions) rather than the scale (magnitude).

RevIN handles Night records with zeroes elegantly because it is instance-based (local) rather than global.
- The paper describes "Affine Parameters" (γ,β) which are learnable weights in the normalization layer
    - Solar curves are not perfectly Gaussian (bell-shaped). The learnable parameters allow the model to slightly "shift" or "stretch" the normalized data to better fit the model's internal preferences.
    - 
"""

class RevIN(nn.Module):
    def __init__(self, num_features: int, eps=1e-5, affine=True):
        """
        :param num_features: the number of features or channels
        :param eps: a value added for numerical stability
        :param affine: if True, RevIN has learnable affine parameters
        """
        super(RevIN, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if self.affine:
            self._init_params()

    def forward(self, x, mode:str):
        if mode == 'norm':
            self._get_statistics(x)
            x = self._normalize(x)
        elif mode == 'denorm':
            x = self._denormalize(x)
        else: raise NotImplementedError
        return x

    def _init_params(self):
        # initialize RevIN params: (C,)
        self.affine_weight = nn.Parameter(torch.ones(self.num_features))
        self.affine_bias = nn.Parameter(torch.zeros(self.num_features))

    def _get_statistics(self, x):
        dim2reduce = tuple(range(1, x.ndim-1))
        self.mean = torch.mean(x, dim=dim2reduce, keepdim=True).detach()
        self.stdev = torch.sqrt(torch.var(x, dim=dim2reduce, keepdim=True, unbiased=False) + self.eps).detach()

    def _normalize(self, x):
        x = x - self.mean
        x = x / self.stdev
        if self.affine:
            x = x * self.affine_weight
            x = x + self.affine_bias
        return x

    def _denormalize(self, x):
        if self.affine:
            x = x - self.affine_bias
            x = x / (self.affine_weight + self.eps*self.eps)
        x = x * self.stdev
        x = x + self.mean
        return x

In [48]:
class moving_avg(torch.nn.Module):
    """
    Moving average block to highlight the trend of time series
    """
    def __init__(self, kernel_size, stride):
        super(moving_avg, self).__init__()
        self.kernel_size = kernel_size
        self.avg = torch.nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # padding on the both ends of time series
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        x = x.permute(0, 2, 1)
        return x
        

class series_decomp(torch.nn.Module):
    """
    Series decomposition block
    """
    def __init__(self, kernel_size):
        super(series_decomp, self).__init__()
        self.moving_avg = moving_avg(kernel_size, stride=1)

    def forward(self, x):
        moving_mean = self.moving_avg(x)
        res = x - moving_mean
        return res, moving_mean




In [ ]:

# --- Dimensions ---
num_past_feats = len(configs.PAST_INPUT_COLS)                     # Number of past features
num_shadows   = len(configs.common_features)                      # Number of shadow features (NWP features)
enc_in        = num_past_feats + num_shadows                 # Number of input features (past + shadow)

print(f"Number of past features: {num_past_feats}")
print(f"Number of shadow features: {num_shadows}")
print(f"Number of input features: {enc_in}")

fut_indices = torch.tensor(
            [configs.FUTURE_INPUT_COLS.index(f) for f in configs.common_features], # Calculate the indices of the shadow features in the future input
            dtype=torch.long
        )
past_target_indices = torch.tensor(
            [configs.PAST_INPUT_COLS.index(f) for f in configs.common_features],  # Calculate the indices of the shadow features in the past input
            dtype=torch.long
        )
# Forward pass
B = x_past.size(0)  # Number of feature in the past sequence input
device = x_past.device
L = configs.seq_len    # L
H = configs.pred_len   # H
print(f"Batch Size (B): {B}, Historical Sequence Length (L): {L}, Future Horizon (H): {H}")

# Ensure indices are on correct device
if fut_indices.device != device:
    fut_indices = fut_indices.to(device)
    past_target_indices = past_target_indices.to(device)

# Shadow columns in x_pred begin after the original past features
shadow_start_idx = num_past_feats # index of the first shadow feature in x_pred = L

# Total length of the sequence (Past + Future), represents time steps.
total_len = L + H

# Initialize the prediction tensor holding (Batch, Time(L+H), Features)
x_pred = torch.zeros(B, total_len, enc_in, device=device)

# A) Fill original past data (history) into original feature slots
x_pred[:, :L, :num_past_feats] = x_past

# B) Extract horizon forecasts for the common/shadow features
# shadow_forecasts: [B, H, S]
shadow_forecasts = torch.index_select(x_future, 2, fut_indices)
# C) Write forecasts into the FUTURE part of the corresponding original columns (proj_to behavior)
# This is equivalent to: x_pred[:, -H:, proj_to] = external_prediction
# Here, proj_to == past_target_indices[i] in your indexing.
for i, past_idx in enumerate(past_target_indices):
    x_pred[:, L:, past_idx] = shadow_forecasts[:, :, i]
    
# D) Shadow columns (future part) directly hold the forecast
x_pred[:, L:, shadow_start_idx:] = shadow_forecasts


# ============================================================
# 2. Standard PowerMamba Processing 
# ============================================================
x = x_pred


# Maintain the distributio drift using RevIN and its inverse.
revin_layer_enc = RevIN(enc_in)
x = revin_layer_enc(x, 'norm')

# Decomposition using Conv 1D kernel
decompsition = series_decomp(configs.kernel_size)
seasonal_init, trend_init = decompsition(x)
x = torch.cat([seasonal_init, trend_init], dim=1)

x = torch.permute(x, (0, 2, 1))
lin1 = nn.Linear(2 * (configs.seq_len + configs.pred_len), configs.seq_len)
x = lin1(x)
print(x.shape)
x = torch.permute(x, (0, 2, 1))
x = revin_layer_enc(x, 'denorm')
print(x.shape)


x = revin_layer_enc(x, 'norm')
seasonal_init, trend_init = decompsition(x)
x_e = torch.cat([seasonal_init, trend_init], dim=1)


x_e = torch.permute(x_e, (0, 2, 1))
lin2 = nn.Linear(2 * configs.seq_len, configs.n_embed)
x_e = lin2(x_e)
lin3 = nn.Linear(4 * configs.n_embed, configs.pred_len)

Number of past features: 2
Number of shadow features: 1
Number of input features: 3
Batch Size (B): 2, Historical Sequence Length (L): 5, Future Horizon (H): 3
torch.Size([2, 3, 5])
torch.Size([2, 5, 3])


In [ ]:

x

tensor([[[ 0.1572,  0.0945,  0.0000],
         [-0.1016,  0.0696,  0.0000],
         [-0.3156,  0.1300,  0.0000],
         [ 0.3624,  0.1567, -0.4922],
         [ 1.1854,  0.0421, -0.8857],
         [-0.8697,  0.1227,  1.2490],
         [-0.4795, -0.1837,  0.4290],
         [ 0.0000, -0.3584, -0.2330],
         [ 0.7667,  1.0473, -0.7576],
         [ 0.7188,  0.9234, -0.7576],
         [ 0.7601,  0.6878, -0.7576],
         [ 0.3220,  0.3659, -0.2653],
         [-0.0549, -0.0782,  0.1282],
         [-0.3971, -0.5906,  0.4545],
         [-0.7874, -1.0438,  0.7809],
         [-1.2669, -1.3854,  1.1073]],

        [[-0.0120, -0.2285,  0.0000],
         [ 0.1948, -0.1402,  0.0000],
         [ 0.2158, -0.0046,  0.0000],
         [-0.3865, -0.0437, -0.3734],
         [ 1.1014,  0.0360, -0.7905],
         [-0.6702,  0.0134,  0.6326],
         [-0.4412,  0.1555,  0.4070],
         [ 0.0000,  0.2179,  0.0974],
         [ 0.9125, -1.2474, -0.7715],
         [ 0.7169, -1.0066, -0.7715],
         [

In [37]:
x_pred

tensor([[[2.4547e-01, 3.3448e+02, 0.0000e+00],
         [2.1110e-01, 3.2526e+02, 0.0000e+00],
         [1.9176e-01, 3.1440e+02, 0.0000e+00],
         [2.1862e-01, 2.9610e+02, 0.0000e+00],
         [2.6863e-01, 2.6147e+02, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[7.9053e-01, 2.4599e+02, 0.0000e+00],
         [7.9466e-01, 2.8589e+02, 0.0000e+00],
         [8.0870e-01, 3.4464e+02, 0.0000e+00],
         [4.2630e-01, 3.9195e+02, 0.0000e+00],
         [8.2144e-01, 4.5753e+02, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00]]])

In [38]:
for i, past_idx in enumerate(past_target_indices):
    x_pred[:, L:, past_idx] = shadow_forecasts[:, :, i]
x_pred

tensor([[[2.4547e-01, 3.3448e+02, 0.0000e+00],
         [2.1110e-01, 3.2526e+02, 0.0000e+00],
         [1.9176e-01, 3.1440e+02, 0.0000e+00],
         [2.1862e-01, 2.9610e+02, 0.0000e+00],
         [2.6863e-01, 2.6147e+02, 0.0000e+00],
         [0.0000e+00, 2.3471e+02, 0.0000e+00],
         [0.0000e+00, 1.8763e+02, 0.0000e+00],
         [0.0000e+00, 1.5563e+02, 0.0000e+00]],

        [[7.9053e-01, 2.4599e+02, 0.0000e+00],
         [7.9466e-01, 2.8589e+02, 0.0000e+00],
         [8.0870e-01, 3.4464e+02, 0.0000e+00],
         [4.2630e-01, 3.9195e+02, 0.0000e+00],
         [8.2144e-01, 4.5753e+02, 0.0000e+00],
         [0.0000e+00, 5.0624e+02, 0.0000e+00],
         [0.0000e+00, 5.6546e+02, 0.0000e+00],
         [0.0000e+00, 6.0189e+02, 0.0000e+00]]])

In [39]:
x_pred[:, L:, shadow_start_idx:] = shadow_forecasts
x_pred

tensor([[[2.4547e-01, 3.3448e+02, 0.0000e+00],
         [2.1110e-01, 3.2526e+02, 0.0000e+00],
         [1.9176e-01, 3.1440e+02, 0.0000e+00],
         [2.1862e-01, 2.9610e+02, 0.0000e+00],
         [2.6863e-01, 2.6147e+02, 0.0000e+00],
         [0.0000e+00, 2.3471e+02, 2.3471e+02],
         [0.0000e+00, 1.8763e+02, 1.8763e+02],
         [0.0000e+00, 1.5563e+02, 1.5563e+02]],

        [[7.9053e-01, 2.4599e+02, 0.0000e+00],
         [7.9466e-01, 2.8589e+02, 0.0000e+00],
         [8.0870e-01, 3.4464e+02, 0.0000e+00],
         [4.2630e-01, 3.9195e+02, 0.0000e+00],
         [8.2144e-01, 4.5753e+02, 0.0000e+00],
         [0.0000e+00, 5.0624e+02, 5.0624e+02],
         [0.0000e+00, 5.6546e+02, 5.6546e+02],
         [0.0000e+00, 6.0189e+02, 6.0189e+02]]])

In [42]:
# which contains (L-H) late-history points + H forecast points.
for i, past_idx in enumerate(past_target_indices):
    shadow_col = shadow_start_idx + i
    x_pred[:, :L, shadow_col] = x_pred[:, -L:, past_idx]

x_pred

tensor([[[2.4547e-01, 3.3448e+02, 2.9610e+02],
         [2.1110e-01, 3.2526e+02, 2.6147e+02],
         [1.9176e-01, 3.1440e+02, 2.3471e+02],
         [2.1862e-01, 2.9610e+02, 1.8763e+02],
         [2.6863e-01, 2.6147e+02, 1.5563e+02],
         [0.0000e+00, 2.3471e+02, 2.3471e+02],
         [0.0000e+00, 1.8763e+02, 1.8763e+02],
         [0.0000e+00, 1.5563e+02, 1.5563e+02]],

        [[7.9053e-01, 2.4599e+02, 3.9195e+02],
         [7.9466e-01, 2.8589e+02, 4.5753e+02],
         [8.0870e-01, 3.4464e+02, 5.0624e+02],
         [4.2630e-01, 3.9195e+02, 5.6546e+02],
         [8.2144e-01, 4.5753e+02, 6.0189e+02],
         [0.0000e+00, 5.0624e+02, 5.0624e+02],
         [0.0000e+00, 5.6546e+02, 5.6546e+02],
         [0.0000e+00, 6.0189e+02, 6.0189e+02]]])

In [47]:
revin_layer_enc = RevIN(enc_in)
x_cop = x_pred.clone()
x_cop = revin_layer_enc(x_cop, 'norm')
x_cop

tensor([[[ 0.9240,  1.1418,  1.7242],
         [ 0.6172,  0.9931,  0.9952],
         [ 0.4445,  0.8179,  0.4320],
         [ 0.6843,  0.5226, -0.5590],
         [ 1.1306, -0.0361, -1.2326],
         [-1.2669, -0.4679,  0.4320],
         [-1.2669, -1.2275, -0.5590],
         [-1.2669, -1.7438, -1.2326]],

        [[ 0.9005, -1.4759, -1.9230],
         [ 0.9116, -1.1468, -0.9722],
         [ 0.9493, -0.6623, -0.2659],
         [-0.0776, -0.2721,  0.5927],
         [ 0.9835,  0.2687,  1.1209],
         [-1.2224,  0.6704, -0.2659],
         [-1.2224,  1.1588,  0.5927],
         [-1.2224,  1.4592,  1.1209]]], grad_fn=<AddBackward0>)

In [57]:
configs.kernel_size = 5

In [58]:
decompsition = series_decomp(configs.kernel_size)
seasonal_init_cop, trend_init_cop = decompsition(x_cop)


In [59]:
seasonal_init_cop, trend_init_cop.shape

(tensor([[[ 0.1572,  0.0945,  0.4042],
          [-0.1016,  0.0696,  0.1319],
          [-0.3156,  0.1300,  0.1600],
          [ 0.3624,  0.1567, -0.5725],
          [ 1.1854,  0.0421, -0.9352],
          [-0.8697,  0.1227,  1.0622],
          [-0.4795, -0.1837,  0.2059],
          [ 0.0000, -0.3584, -0.4676]],
 
         [[-0.0120, -0.2285, -0.5216],
          [ 0.1948, -0.1402, -0.0739],
          [ 0.2158, -0.0046,  0.0236],
          [-0.3865, -0.0437,  0.5508],
          [ 1.1014,  0.0360,  0.7660],
          [-0.6702,  0.0134, -0.8982],
          [-0.4412,  0.1555, -0.1452],
          [ 0.0000,  0.2179,  0.3830]]], grad_fn=<SubBackward0>),
 torch.Size([2, 8, 3]))

In [62]:
x_cop = torch.cat([seasonal_init_cop, trend_init_cop], dim=1)

In [63]:
x_cop.shape

torch.Size([2, 16, 3])

In [65]:
x_cop = torch.permute(x_cop, (0, 2, 1))


In [66]:
x_cop.shape

torch.Size([2, 3, 16])

In [68]:
lin1 = nn.Linear(2 * (configs.seq_len + configs.pred_len), configs.seq_len)
x_cop = lin1(x_cop)

In [69]:
x_cop.shape

torch.Size([2, 3, 5])

In [72]:
x_cop = torch.permute(x_cop, (0, 2, 1))
x_cop.shape

torch.Size([2, 3, 5])